# Exercise 8: The crash report, live

A Spark Structured Streaming query reads the log records of the topic `logs`, computes the crash report
(users with more than 2 severe crashes per 10-second interval) and writes alerts into the topic
`crash_alerts`. Two teams read them:

| Team | Needs |
|---|---|
| on-call engineer | an alert at most **20 s** after the interval ends, for 95 % of the problems (p95) |
| operations team | a **correct** report (at least 99 % of the rows), late records included |

All the code is in this notebook: the Spark query (sections 1-3) and the two teams' readers (section 4).
`tools/streamlib.py` is only the test bench: it writes live records, simulates a crash and checks what
each team gets. You write one function, `start_query()` (architecture C: also `ops_report()`), and look
inside Spark (section 8). Save your final version of this notebook into your `solution/` folder.

**Keep the Spark UI open while you work: <http://localhost:4040> → *Structured Streaming*.** It shows
what your query really does, micro-batch by micro-batch: how many records each one reads, how long it
takes, where the watermark is, how much state Spark keeps and how many late records it drops. Most
questions of this exercise are answered there. It exists once the next cell has started Spark; reload
the page to see new micro-batches.

In [ ]:
import sys; sys.path.append("/home/jovyan/tools")
from streamlib import check_all, show_alerts, produce, reset, crash_point, load_lake, read_lake   # the test bench

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("Exercise8")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.1")   # Kafka connector
         .config("spark.sql.shuffle.partitions", "4")       # small data: 4 tasks per shuffle, not 200
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

KAFKA = "kafka:29092"                                       # Kafka, seen from this container
CHECKPOINT = "/home/jovyan/data/stream/checkpoint"          # check_all and reset() delete it

## 1. Read: the topic as an unbounded table

A stream is a table that never ends. Spark runs the query in **micro-batches**: each one reads the
records that arrived since the last one, updates the result and writes what changed. By default the
next micro-batch starts as soon as the previous one is done. Each micro-batch is a small Exercise 7 run:
read the new offsets, process, write, then note where it stopped (in the *checkpoint*).

Every record carries its **event time** (when it happened on the phone), which is not the time it
arrives (*processing time*). Late records have an event time in the past.

You will see the micro-batches in the Spark UI (<http://localhost:4040> → *Structured Streaming* → the
*Run ID* of your query) as soon as the example runs in section 5.

In [ ]:
SCHEMA = "timestamp LONG, status STRING, severity STRING, source_ip STRING, user_id STRING, content STRING"

def read_stream():
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", KAFKA)
           .option("subscribe", "logs")
           .option("startingOffsets", "latest")     # a new query starts at the end; a restarted one where it stopped
           .load())                                 # columns: key, value (bytes), partition, offset, timestamp, ...
    log = F.from_json(F.col("value").cast("string"), SCHEMA)
    return (raw.select(log.alias("log")).select("log.*")
            .withColumn("event_time", F.timestamp_millis("timestamp")))   # when it happened, not when it arrived

## 2. The crash report: the same code for a stream and a batch

`groupBy(window(...))` keeps one count per 10-second interval and user. For a stream, Spark keeps these
counts between micro-batches (the *state*, saved with the checkpoint). The **watermark**
(`withWatermark`, section 5) tells Spark how long to wait for late records before an interval is
complete and its state can be dropped.

In [ ]:
def crash_report(records):
    return (records
            .filter(F.lower("content").contains("crash") & F.col("severity").isin("High", "Critical"))
            .groupBy(F.window("event_time", "10 seconds").alias("interval"), "user_id")
            .agg(F.count("*").alias("crash_count"))
            .filter("crash_count > 2"))

## 3. Write: alerts into Kafka

The **output mode** decides which rows a micro-batch writes:

- `"append"`: every row once, when its interval is complete (after the watermark);
- `"update"`: a row as soon as it exists, and a new version whenever its count changes.

`foreachBatch` gets each micro-batch as an ordinary DataFrame; we write it to the topic `crash_alerts`
with the key *interval + user*. Only after `write_batch` returns does Spark note the micro-batch in the
checkpoint as done. `crash_point` marks the moment in between: `check_all` simulates a crash there once.

In [ ]:
def write_alerts(report, mode):
    start = F.date_format("interval.start", "yyyy-MM-dd HH:mm:ss")
    end = F.date_format("interval.end", "yyyy-MM-dd HH:mm:ss")
    alerts = report.select(
        F.concat_ws(" ", start, "user_id").alias("key"),          # one key per interval and user
        F.to_json(F.struct(start.alias("interval_start"), end.alias("interval_end"),
                           "user_id", "crash_count")).alias("value"))

    def write_batch(batch, batch_id):                             # once per micro-batch
        (batch.write.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("topic", "crash_alerts")
         .save())
        crash_point(batch_id)                                     # written to Kafka, not yet in the checkpoint

    return (alerts.writeStream.queryName("crash_alerts")
            .outputMode(mode)
            .foreachBatch(write_batch)
            .option("checkpointLocation", CHECKPOINT)             # where it stopped + the state
            .start())                                             # runs in the background; returns the query

## 4. Read the alerts: the two teams

Each team reads the messages of `crash_alerts`, oldest first, as `(key, crash_count)`; the key is
*interval + user*. A message can come more than once: as an **update** (same key, new count, in output
mode `update`) or as a **repeat** (same key, same count: the micro-batch written again after a crash).
`check_all` passes the messages to these two functions and reports what each team got.

In [ ]:
def pager(messages):                  # the on-call engineer: one page per problem
    paged = {}
    for key, count in messages:
        if key not in paged:          # a later message for the same key: no new page
            paged[key] = count
    return paged

def ops_view(messages):               # the operations team: the latest count per problem
    return {key: count for key, count in messages}

## 5. The example: a stream that does not wait

It waits only 5 s for late records (watermark), then writes each interval once (append).

`check_all` runs it on 2 minutes of live log records (5 % of them late, by up to 30 s), stops it once at
the crash point, and restarts it from its checkpoint. It takes about 3 minutes.
While it runs, watch it from both sides:
- **Spark UI** (<http://localhost:4040> → *Structured Streaming* → click the *Run ID*): the micro-batches
  as they happen. Reload now and then: *Input Rows* and *Batch Duration* per micro-batch, the *Global
  Watermark Gap* (how far the watermark is behind), the *State Rows* (the open intervals), and *Late Rows
  Dropped By Watermark*. At half time the run fails (the simulated crash) and a new run of the same query
  starts.
- **Kafka UI** (<http://localhost:8080> → Topics → `crash_alerts` → Messages): the alerts as they arrive.

In [ ]:
def start_query():
    records = read_stream().withWatermark("event_time", "5 seconds")   # wait 5 s for late records
    return write_alerts(crash_report(records), mode="append")          # every row once, when its interval is complete

check_all(start_query, pager, ops_view)

In [ ]:
show_alerts(10)   # the last 10 alerts (times in UTC)
# Compare "written at" with the interval: why do all rows of an interval arrive together,
# and why about 7 s after its end?

## 6. Your architecture

Write `start_query()` for the architecture you chose (A, B or C, see the README): with sections 1-4 it
is a watermark and an output mode. Architecture C also needs `ops_report()`: the report for the
operations team, computed in batch over the lake of Exercise 7 (`load_lake()` runs Exercise 7's job,
`read_lake()` reads the lake, and `crash_report()` works on it unchanged).

Run your check with the Spark UI open, as in the example: compare its watermark gap, state rows and
late rows dropped with the example's (both are in the list as long as the kernel runs).

`produce(30)` writes 30 s of live records into `logs` if you want to try something by hand;
`reset()` stops the query and deletes its checkpoint.

In [ ]:
def start_query():
    records = read_stream().withWatermark("event_time", ...)
    return write_alerts(crash_report(records), mode=...)

# only for architecture C
def ops_report():
    ...

## 7. Check it

`check_all` prints what each team got, compared with **the truth**: the problems counted over every record
it wrote, late ones included.

- **On-call engineer**: the **delay** of each page after its interval ended (negative: paged before the
  interval was over). **p50** (the median): half of the pages came faster; **p95**: 95 % came faster. The
  requirement uses p95: the average hides the slow pages, and the maximum would be decided by a single one.
  **Messages → pages**: the pager ignored *repeats* (written again after the restart) and *updates* (a new
  count for the same problem).
- **Operations team**: the report rows that are right (the right problem with the right count), and the
  wrong ones: *missing*, *wrong count* or *extra*. The examples show, for a few wrong rows, how late their
  crashes arrived.
- Architecture C only, **alerts vs report**: problems in the report that were never paged or paged with
  another count.
- **Result**: which team is served.

In [ ]:
check_all(start_query, pager, ops_view)
# architecture C: check_all(start_query, pager, ops_report=ops_report)

## 8. Look inside: the Spark UI

Right after your `check_all`, open <http://localhost:4040> (4041 if 4040 is taken) → **Structured
Streaming**. The UI lives as long as this notebook's kernel: take the screenshots **before** you restart
the kernel or close the notebook.

1. **The query list** (*Completed Streaming Queries*): your check made two runs of `crash_alerts`. Compare
   their *ID* and *Run ID*. What did the restart reuse from the checkpoint? Screenshot: `ui-runs.png`.
   (The run marked FAILED is the simulated crash: its *Error* column shows Spark's report of it.)
2. Click the newest run's *Run ID* → statistics. **Batch Duration** and **Global Watermark Gap**: how long
   is a micro-batch, and how far behind real time is the watermark? How does that explain your alert
   delay? Screenshot: `ui-delay.png`.
3. Further down: **Aggregated Number Of Total State Rows** and **Aggregated Number Of Late Rows Dropped
   By Watermark**: what does your query keep, and what does it lose? Screenshot: `ui-state.png`.

The example's runs are in the same list (as long as the kernel was not restarted): compare them with
yours if you like.